## Initial Recording Code

In [5]:
import ipywidgets.widgets as widgets
from IPython.display import display
import motors

#create two widgets for the displaying of the image
display_color = widgets.Image(format='jpeg', width='45%') #determine the width of the color image
display_depth = widgets.Image(format='jpeg', width='45%')  #determine the width of the depth image
layout=widgets.Layout(width='100%')

sidebyside = widgets.HBox([display_color, display_depth],layout=layout) #horizontal display

#Convert a NumPy array to JPEG-encoded data for display
def bgr8_to_jpeg(value):
    return bytes(cv2.imencode('.jpg',value)[1])
    
# display the widget
display(sidebyside) 

#code modified from https://www.stereolabs.com/docs/video/recording
import pyzed.sl as sl
import cv2

# Create a ZED camera object
zed = sl.Camera()

# Create a InitParameters object and set configuration parameters
init = sl.InitParameters()
init.camera_resolution = sl.RESOLUTION.HD720 #VGA(672*376), HD720(1280*720), HD1080 (1920*1080) or ...
init.depth_mode = sl.DEPTH_MODE.ULTRA  # Use ULTRA depth mode
init.coordinate_units = sl.UNIT.MILLIMETER  # Use meter units (for depth measurements)

#open zed camera
status = zed.open(init) 
if status != sl.ERROR_CODE.SUCCESS: 
    print("Zed Open", status, "Exit program.")
    exit(1)

#save the data in svo2 format (highly recommended)
recording_param = sl.RecordingParameters('test.svo2', sl.SVO_COMPRESSION_MODE.LOSSLESS) #jetson orin nano only support LOSSLESS ...
err = zed.enable_recording(recording_param)
if err != sl.ERROR_CODE.SUCCESS:
    print("Recording ZED : ", err)
    exit(1)

# Create and set RuntimeParameters after opening the camera
runtime = sl.RuntimeParameters()
print("Recording") 

# Declare your sl.Mat matrices
image_zed = sl.Mat()
depth_zed = sl.Mat()

#Record 180 frames
Total_frame = 180
i = 0 #frame index
t1 = cv2.getTickCount() #keep a record of the time, fps = frames/total time
while i < Total_frame:
    if zed.grab(runtime) == sl.ERROR_CODE.SUCCESS : # Check that a new image is successfully acquired
        i += 1 
        # The following code is for display purposes. You can comment it out to achieve higher recording FPS.
        zed.retrieve_image(image_zed, sl.VIEW.LEFT)
        zed.retrieve_measure(depth_zed, sl.MEASURE.DEPTH)

        scale = 0.1 # reduce image size
        color_value = image_zed.get_data()
        color_rgb_value = color_value[:, :, :3] #RGBA -> RGB
        resized_color = cv2.resize(color_rgb_value, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
        display_color.value = bgr8_to_jpeg(resized_color)
        
        #Get the depth image and convert it into a color image (for display only)
        depth_image = depth_zed.get_data()
        depth_colormap = cv2.applyColorMap(cv2.convertScaleAbs(depth_image, alpha=0.03), cv2.COLORMAP_JET) 
        resized_depth = cv2.resize(depth_colormap, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
        display_depth.value = bgr8_to_jpeg(resized_depth)

zed.close()
total_time = (cv2.getTickCount() - t1) / cv2.getTickFrequency()
print('time, ', total_time, 'fps ', Total_frame/ total_time)


#TODO - add in driving funcitonality and the stopping of frcording on button press and save the movement commands as a CSV so that a
#command can be used to reference in the training of an AI model
#widgets for the input
text_input = widgets.Text(value='', placeholder='Type something', description='Input:', disabled=False)

def on_text_change(change):
    input_value = change['new'] #get the text 
    if(input_value[-1] == 'w'): #only use the last letter
        robot.forward(0.5) 
    elif(input_value[-1] == 's'):
        robot.backward(0.5) 
    elif(input_value[-1] == 'a'):
        robot.right(0.5) 
    elif(input_value[-1] == 'd'):
        robot.left(0.5) 
    else:
        robot.stop() 

text_input.observe(on_text_change, names='value')
display(text_input)

[2026-03-04 11:40:11 UTC][ZED][INFO] Logging level INFO
[2026-03-04 11:40:11 UTC][ZED][INFO] Logging level INFO
Recording
[2026-03-04 11:40:11 UTC][ZED][INFO] Logging level INFO
[2026-03-04 11:40:12 UTC][ZED][INFO] [Init]  Depth mode: ULTRA
[2026-03-04 11:40:13 UTC][ZED][INFO] [Init]  Camera successfully opened.
[2026-03-04 11:40:13 UTC][ZED][INFO] [Init]  Camera FW version: 1523
[2026-03-04 11:40:13 UTC][ZED][INFO] [Init]  Video mode: HD720@60
[2026-03-04 11:40:13 UTC][ZED][INFO] [Init]  Serial Number: S/N 39315162
[2026-03-04 11:40:14 UTC][ZED][INFO] [Init]  Notice: The recording is using SVO version 2, enabled by default starting from SDK version 4.1. To revert to the original SVO version, set the environment variable "ZED_SDK_SVO_VERSION" to 1
time,  14.512314598 fps  12.403259230943528


## Recording on the move

In [2]:
import ipywidgets.widgets as widgets
from IPython.display import display
import pyzed.sl as sl
import cv2
import csv
import time
import os
import threading  # New import for threading

# --- MOTOR SETUP ---
import motors
robot = motors.MotorsYukon(mecanum=False)

# --- UI WIDGETS ---
display_color = widgets.Image(format='jpeg', width='45%')
display_depth = widgets.Image(format='jpeg', width='45%')
text_input = widgets.Text(placeholder='w/a/s/d to drive, q to quit', description='Drive:')
stop_button = widgets.Button(description="Stop Recording", button_style='danger')

layout = widgets.Layout(width='100%')
sidebyside = widgets.HBox([display_color, display_depth], layout=layout)
display(sidebyside)
display(text_input)
display(stop_button)

# --- GLOBAL STATE ---
is_recording = True
current_command = "stop"
frame_idx = 0

def on_text_change(change):
    global current_command, is_recording
    val = change['new']
    if not val: return
    
    cmd = val[-1].lower()
    if cmd == 'w':
        robot.forward(0.2)
        current_command = "forward"
    elif cmd == 's':
        robot.backward(0.2)
        current_command = "backward"
    elif cmd == 'a':
        robot.left(0.2)
        current_command = "left"
    elif cmd == 'd':
        robot.right(0.2)
        current_command = "right"
    elif cmd == 'q':
        is_recording = False
    elif cmd == 'W':     #speed boost
        robot.forward(0.5)
    else:
        robot.stop()
        current_command = "stop"

def stop_clicked(b):
    global is_recording
    is_recording = False

text_input.observe(on_text_change, names='value')
stop_button.on_click(stop_clicked)

# --- ZED SETUP ---
def bgr8_to_jpeg(value):
    return bytes(cv2.imencode('.jpg', value)[1])

zed = sl.Camera()
init = sl.InitParameters()
init.camera_resolution = sl.RESOLUTION.HD1080
init.depth_mode = sl.DEPTH_MODE.ULTRA
init.coordinate_units = sl.UNIT.MILLIMETER

if zed.open(init) != sl.ERROR_CODE.SUCCESS:
    print("Zed Open Failed")
    exit(1)

recording_param = sl.RecordingParameters('training_data_2.svo2', sl.SVO_COMPRESSION_MODE.LOSSLESS)
zed.enable_recording(recording_param)

# --- THREADED RECORDING FUNCTION ---
def recording_loop():
    global is_recording, frame_idx, current_command
    
    image_zed = sl.Mat()
    depth_zed = sl.Mat()
    runtime = sl.RuntimeParameters()
    
    # Open CSV inside the thread
    with open('movement_log.csv', mode='w', newline='') as csv_file:
        log_writer = csv.writer(csv_file)
        log_writer.writerow(['timestamp', 'frame_index', 'command'])
        
        t_start = cv2.getTickCount()
        
        while is_recording:
            if zed.grab(runtime) == sl.ERROR_CODE.SUCCESS:
                # 1. Log Command
                log_writer.writerow([time.time(), frame_idx, current_command])
                
                # 2. Retrieve for Display
                zed.retrieve_image(image_zed, sl.VIEW.LEFT)
                zed.retrieve_measure(depth_zed, sl.MEASURE.DEPTH)

                # 3. Update Widgets (Scale down for speed)
                color_data = image_zed.get_data()[:, :, :3]
                resized_color = cv2.resize(color_data, None, fx=0.1, fy=0.1)
                display_color.value = bgr8_to_jpeg(resized_color)

                depth_data = depth_zed.get_data()
                depth_vis = cv2.applyColorMap(cv2.convertScaleAbs(depth_data, alpha=0.03), cv2.COLORMAP_JET)
                resized_depth = cv2.resize(depth_vis, None, fx=0.1, fy=0.1)
                display_depth.value = bgr8_to_jpeg(resized_depth)

                frame_idx += 1
        
        # Cleanup inside thread once is_recording is False
        robot.stop()
        zed.disable_recording()
        zed.close()
        total_time = (cv2.getTickCount() - t_start) / cv2.getTickFrequency()
        print(f"\nStopped. Saved {frame_idx} frames in {total_time:.2f}s (FPS: {frame_idx/total_time:.2f})")

# --- START THE THREAD ---
print("Starting background recording thread...")
rec_thread = threading.Thread(target=recording_loop)
rec_thread.start()

Text(value='', description='Drive:', placeholder='w/a/s/d to drive, q to quit')

Button(button_style='danger', description='Stop Recording', style=ButtonStyle())

[2026-03-04 12:35:28 UTC][ZED][INFO] Logging level INFO
[2026-03-04 12:35:28 UTC][ZED][INFO] Logging level INFO
Starting background recording thread...
[2026-03-04 12:35:28 UTC][ZED][INFO] Logging level INFO
[2026-03-04 12:35:29 UTC][ZED][INFO] [Init]  Depth mode: ULTRA
[2026-03-04 12:35:30 UTC][ZED][INFO] [Init]  Camera successfully opened.
[2026-03-04 12:35:30 UTC][ZED][INFO] [Init]  Camera FW version: 1523
[2026-03-04 12:35:30 UTC][ZED][INFO] [Init]  Video mode: HD1080@30
[2026-03-04 12:35:30 UTC][ZED][INFO] [Init]  Serial Number: S/N 39315162
[2026-03-04 12:35:31 UTC][ZED][INFO] [Init]  Notice: The recording is using SVO version 2, enabled by default starting from SDK version 4.1. To revert to the original SVO version, set the environment variable "ZED_SDK_SVO_VERSION" to 1
